1. Load packages, input files, and diagnosis and covariate data. Rename covariates 

In [2]:
# Step 1: Load packages
import pandas as pd
import numpy as np

# Step 2: Load input files
base_path = "/static/PMBB/PMBB-Release-2026-4.0/Phenotype/4.0/"
COV_file = base_path + "PMBB-Release-2026-4.0_phenotype_covariates.txt"
DX_file = base_path + "PMBB-Release-2026-4.0_phenotype_condition_occurrence.txt"
# Don't need to get rid of unrelated because SAIGE can account for relationships 
# UNREL_FILE = "/static/PMBB/PMBB-Release-2026-4.0/Exome/relationships/PMBB-Release-2026-4.0_genetic_exome.2nd_degree_unrelated.txt"

# Step 3: Load diagnosis and covariate data
cols = pd.read_csv(DX_file, sep="\t", nrows=5)
DX_v4 = pd.read_csv(
    DX_file,
    sep="\t",
    usecols=["person_id", "condition_concept_id", "condition_start_date"]
)
print(DX_v4.columns)
covars = pd.read_csv(COV_file, sep="\t")
print(covars.columns)

# Step 4: Recode and rename covariates
covars.columns = covars.columns.str.strip()  # prevents hidden whitespace bugs

covars['SEX'] = covars['sequenced_gender'].replace({'Male': 1, 'Female': 2})

covars = covars.rename(columns={
    "person_id": "PMBB_ID",
    "sample_age": "AGE"
})

Index(['person_id', 'condition_concept_id', 'condition_start_date'], dtype='str')
Index(['person_id', 'batch', 'crep_highrisk_flag', 'sequenced_gender',
       'sample_date', 'sample_age', 'exome_PC1', 'exome_PC2', 'exome_PC3',
       'exome_PC4', 'exome_PC5', 'exome_PC6'],
      dtype='str')


2. Define phecodes, select only 3 columns (the person ID, phecode, and diagnosis date), define cases (and/or for hearing loss and tinnitus, rule of 2), define controls (anyone who is not a case), filter unrelated samples, make table with only the controls and cases (which includes everyone because the control definition is very liberal) 

In [1]:
# Step 1: Load packages
import pandas as pd
import numpy as np

# Step 2: Load input files
base_path = "/static/PMBB/PMBB-Release-2026-4.0/Phenotype/4.0/"
COV_file = base_path + "PMBB-Release-2026-4.0_phenotype_covariates.txt"
DX_file = base_path + "PMBB-Release-2026-4.0_phenotype_condition_occurrence.txt"
# Don't need to get rid of unrelated because SAIGE can account for relationships 
# UNREL_FILE = "/static/PMBB/PMBB-Release-2026-4.0/Exome/relationships/PMBB-Release-2026-4.0_genetic_exome.2nd_degree_unrelated.txt"

# Step 3: Load diagnosis and covariate data
cols = pd.read_csv(DX_file, sep="\t", nrows=5)
DX_v4 = pd.read_csv(
    DX_file,
    sep="\t",
    usecols=["person_id", "condition_concept_id", "condition_start_date"]
)
print(DX_v4.columns)
covars = pd.read_csv(COV_file, sep="\t")
print(covars.columns)

# Step 4: Recode and rename covariates
covars.columns = covars.columns.str.strip()  # prevents hidden whitespace bugs

covars['SEX'] = covars['sequenced_gender'].replace({'Male': 1, 'Female': 2})

covars = covars.rename(columns={
    "person_id": "PMBB_ID",
    "sample_age": "AGE"
})
HL_ICD = ["H90", "H91", "H93"]     # hearing loss
TIN_ICD = ["H93.1"]        # tinnitus

# -----------------------------
# 0) Harmonize ID types
# -----------------------------
DX = DX_v4[['person_id', 'condition_concept_id', 'condition_start_date']].copy()
DX['person_id'] = DX['person_id'].astype(str)

phenos = covars[['PMBB_ID', 'batch', 'AGE', 'sequenced_gender']].copy()
phenos['PMBB_ID'] = phenos['PMBB_ID'].astype(str)

# -----------------------------
# 1) Define HL and tinnitus cases (RULE OF 2)
# -----------------------------
hl_dx = DX[DX['condition_concept_id'].astype(str).str.contains("|".join(HL_ICD), na=False)]

hl_counts = (
    hl_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'HL_count'
        })
)

hl_counts['HL_case'] = (hl_counts['HL_count'] >= 2).astype(int)

phenos = phenos.merge(
    hl_counts[['PMBB_ID', 'HL_case', 'HL_count']],
    how='left',
    on='PMBB_ID'
)
phenos['HL_case'] = phenos['HL_case'].fillna(0).astype(int)
phenos['HL_count'] = phenos['HL_count'].fillna(0).astype(int)

tin_dx = DX[DX['condition_source_value'].astype(str).str.contains("|".join(TIN_ICD), na=False)]

tin_counts = (
    tin_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'TIN_count'
        })
)

tin_counts['TIN_case'] = (tin_counts['TIN_count'] >= 2).astype(int)

phenos = phenos.merge(
    tin_counts[['PMBB_ID', 'TIN_case', 'TIN_count']],
    how='left',
    on='PMBB_ID'
)
phenos['TIN_case'] = phenos['TIN_case'].fillna(0).astype(int)
phenos['TIN_count'] = phenos['TIN_count'].fillna(0).astype(int)

# -----------------------------
# 2) Filter to unrelated; NOT NEEDED IF USING SAIGE 
# -----------------------------
#unrel_ids = pd.read_csv(UNREL_FILE, header=None)[0].astype(str).unique()
#phenos = phenos[phenos['PMBB_ID'].isin(unrel_ids)].copy()

# -----------------------------
# 3) Define case and control
# -----------------------------
phenos['CASE'] = ((phenos['HL_case'] == 1) | (phenos['TIN_case'] == 1)).astype(int)

phenos['CONTROL'] = np.where(
    (phenos['CASE'] == 0),
    1, 
    0
)

# -----------------------------
# 4) Final binary case-control dataset
#    Keep ONLY cases and  controls
# -----------------------------
phenos_cc = phenos[
    (phenos['CASE'] == 1) | (phenos['CONTROL'] == 1)
].copy()
# Keeps everyone who is a case or a control (which is basically everyone)

# Final phenotype label: 1=case, 0=control
phenos_cc['PHENO'] = phenos_cc['CASE']

# -----------------------------
# 5) Summary
# -----------------------------

print("Hearing loss + tinnitus ExWAS phenotype")
print("Total:", len(phenos_cc))
print("Cases:", phenos_cc['PHENO'].sum())
print("Controls:", (phenos_cc['PHENO'] == 0).sum())

# -----------------------------
# 6) Save outputs
# -----------------------------
outpath = "/project/hall/analysis/hearing-loss-genomics/elena/rare_variant/HL_TIN_exWAS_PMBBv4_final"

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.csv", index=False
)

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.txt", sep='\t', index=False
)


Index(['person_id', 'condition_concept_id', 'condition_start_date'], dtype='str')
Index(['person_id', 'batch', 'crep_highrisk_flag', 'sequenced_gender',
       'sample_date', 'sample_age', 'exome_PC1', 'exome_PC2', 'exome_PC3',
       'exome_PC4', 'exome_PC5', 'exome_PC6'],
      dtype='str')


: 

3. Creates a file for PLINK (uses FID and IID instead of the PMBB IDs)

In [ ]:
###Make keep file for plink###
# Load final phenotype table
pheno_file = "/project/hall/analysis/hearing-loss-genomics/elena/rare_variant/HL_TIN_exWAS_PMBBv4_final.csv"
phenos = pd.read_csv(pheno_file)


# Extract FID and IID (both = PMBB_ID)
keep_df = phenos[['PMBB_ID']].copy()
keep_df['FID'] = keep_df['PMBB_ID']
keep_df['IID'] = keep_df['PMBB_ID']


# Reorder columns
keep_df = keep_df[['FID', 'IID']]


# Save as keep file
keep_file = "/project/hall/analysis/hearing-loss-genomics/elena/rare_variant/HL_TIN_PMBBv4_keep.txt"
keep_df.to_csv(keep_file, sep='\t', index=False, header=False)


print(f"PLINK keep file written to: {keep_file}")

4. QC check of PMBB data, checks how many of the samples actually have genotype data, makes new file with that data

In [ ]:
import pandas as pd

# File paths
phenotype_path = "/project/hall/analysis/hearing-loss-genomics/elena/rare_variant/HL_TIN_exWAS_PMBBv4_final.txt"
fam_path = "/static/PMBB/PMBB-Release-2026-4.0/Imputed/common_snps_LD_pruned/PMBB-Release-2026-4.0_genetic_imputed.commonsnps.ldpruned.ALL.fam"
output_path = "/project/hall/analysis/hearing-loss-genomics/elena/rare_variant/pheno_PMBBv4_final_samplelist_filtered.txt"

# Load PLINK FAM file
fam_df = pd.read_csv(fam_path, sep=r"\s+", header=None)
fam_df.columns = ["FID", "IID", "Father", "Mother", "Sex", "Phenotype"]

# Load phenotype file
pheno_df = pd.read_csv(phenotype_path, sep="\t")

# Make sure IDs are strings
fam_df["IID"] = fam_df["IID"].astype(str)
pheno_df["PMBB_ID"] = pheno_df["PMBB_ID"].astype(str)

# Keep only samples with genotype data
filtered_df = pheno_df[
    pheno_df["PMBB_ID"].isin(fam_df["IID"])
]

# Count
num_cases = (filtered_df['PHENO'] == 1).sum()
num_controls = (filtered_df['PHENO'] == 0).sum()

# Save to file
filtered_df.to_csv(output_path, sep='\t', index=False)

# Output counts
print(f"Saved filtered file to: {output_path}")
print(f"Cases: {num_cases}")
print(f"Controls: {num_controls}")